# Supplement — Proving Logistic-Regression Gradients with Two Input Features

**CSS371: Natural Language Processing**  
**Goal:** follow every number and derivative for $x_1,x_2$ before using matrix notation

We use one sigmoid neuron with exactly two inputs, two weights, and one bias:

$$z=x_1w_1+x_2w_2+b,\qquad p=\hat y=\sigma(z).$$

![Two-feature logistic-regression network](assets/logistic_regression_network.svg)

The notebook first proves the gradient for one example, checks it numerically, performs one parameter update, and then extends the same calculation to a mini-batch of two examples.


## Learning outcomes

Students should be able to:

1. calculate a two-feature logit by hand;
2. derive $dw_1$, $dw_2$, and $db$;
3. explain why each input scales its corresponding weight gradient;
4. perform one Gradient Descent update;
5. derive the two-example matrix formula $dW=X^\top dZ$;


In [1]:
import sys
import numpy as np
from IPython.display import HTML, display

np.set_printoptions(precision=8, suppress=True)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def bce(y, p, epsilon=1e-12):
    p = np.clip(p, epsilon, 1.0-epsilon)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))


Python 3.13.15 | NumPy 2.1.3


# Part I — One example, two input features

## 1. Assign concrete values

Use

$$x_1=2,\qquad x_2=-1,$$
$$w_1=0.30,\qquad w_2=-0.20,\qquad b=0.10,$$
$$y=1.$$

The label $y=1$ says that this example belongs to the positive class. The current parameters determine how confident the model is.

### Shape view

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix}_{ 1 \times2 },\qquad
\mathbf{w}=\begin{bmatrix}0.30\\-0.20\end{bmatrix}_{2\times1}.$$

Therefore $\mathbf{xw}$ has shape $(1\times2)(2\times1)=(1\times1)$.


In [2]:
x1, x2 = 2.0, -1.0
w1, w2, bias = 0.30, -0.20, 0.10
y = 1.0

z = x1*w1 + x2*w2 + bias
p = float(sigmoid(z))
loss = float(bce(np.array([y]), np.array([p])))

print(f"z = ({x1})({w1}) + ({x2})({w2}) + {bias}")
print(f"z = {x1*w1:.2f} + {x2*w2:.2f} + {bias:.2f} = {z:.6f}")
print(f"p = sigmoid(z) = {p:.8f}")
print(f"loss = -log(p) = {loss:.8f}")


z = (2.0)(0.3) + (-1.0)(-0.2) + 0.1
z = 0.60 + 0.20 + 0.10 = 0.900000
p = sigmoid(z) = 0.71094950
loss = -log(p) = 0.34115387


## 2. Prove the sigmoid derivative

Let

$$p=\sigma(z)=\frac{1}{1+e^{-z}}=(1+e^{-z})^{-1}.$$

Differentiate using the power rule and chain rule:

$$\frac{dp}{dz}
=-(1+e^{-z})^{-2}\frac{d}{dz}(1+e^{-z}).$$

Since $d(e^{-z})/dz=-e^{-z}$,

$$\frac{dp}{dz}=\frac{e^{-z}}{(1+e^{-z})^2}.$$

Now write

$$p=\frac{1}{1+e^{-z}},\qquad 1-p=\frac{e^{-z}}{1+e^{-z}}.$$

Multiplying them gives

$$p(1-p)=\frac{e^{-z}}{(1+e^{-z})^2}.$$

Therefore

$$\boxed{\frac{dp}{dz}=p(1-p)}.$$

For our numerical example:

$$\frac{dp}{dz}=0.7109495(1-0.7109495)\approx0.2055003.$$


In [3]:
dp_dz = p*(1-p)
epsilon = 1e-5
numeric_dp_dz = (sigmoid(z+epsilon)-sigmoid(z-epsilon))/(2*epsilon)
print(f"analytical dp/dz = p(1-p) = {dp_dz:.10f}")
print(f"numerical  dp/dz          = {numeric_dp_dz:.10f}")
print(f"absolute difference       = {abs(dp_dz-numeric_dp_dz):.2e}")


analytical dp/dz = p(1-p) = 0.2055003073
numerical  dp/dz          = 0.2055003073
absolute difference       = 4.69e-12


## 3. Prove the BCE derivative

For one example,

$$\ell(y,p)=-y\log p-(1-y)\log(1-p).$$

Differentiate with respect to $p$:

$$\frac{\partial\ell}{\partial p}
=-\frac{y}{p}+\frac{1-y}{1-p}.$$

Use common denominator $p(1-p)$:

$$\frac{\partial\ell}{\partial p}
=\frac{-y(1-p)+(1-y)p}{p(1-p)}.$$

Expand the numerator:

$$-y+yp+p-yp=p-y.$$

Thus

$$\boxed{\frac{\partial\ell}{\partial p}=\frac{p-y}{p(1-p)}}.$$

Because our label is $y=1$, this also reduces directly to

$$\ell=-\log p,\qquad\frac{\partial\ell}{\partial p}=-\frac{1}{p}.$$


In [4]:
dL_dp_general = (p-y)/(p*(1-p))
dL_dp_y1 = -1/p
numeric_dL_dp = (bce(np.array([y]), np.array([p+epsilon])) -
                  bce(np.array([y]), np.array([p-epsilon])))/(2*epsilon)
print(f"general formula (p-y)/(p(1-p)) = {dL_dp_general:.10f}")
print(f"because y=1, -1/p              = {dL_dp_y1:.10f}")
print(f"numerical derivative           = {numeric_dL_dp:.10f}")


general formula (p-y)/(p(1-p)) = -1.4065696597
because y=1, -1/p              = -1.4065696597
numerical derivative           = -1.4065696598


## 4. Chain the derivatives: why the result is $p-y$

The dependency path is

$$z\longrightarrow p=\sigma(z)\longrightarrow\ell(y,p).$$

By the chain rule,

$$\frac{\partial\ell}{\partial z}
=\frac{\partial\ell}{\partial p}\frac{dp}{dz}.$$

Substitute the results:

$$\frac{\partial\ell}{\partial z}
=\frac{p-y}{p(1-p)}\;p(1-p).$$

The factors cancel:

$$\boxed{\frac{\partial\ell}{\partial z}=p-y}.$$

Numerically,

$$d z=0.7109495-1=-0.2890505.$$

The negative sign is meaningful: the predicted probability is below the positive target, so reducing loss requires increasing the logit. Gradient Descent subtracts a negative value and therefore moves the logit upward.


In [5]:
dL_dz_chain = dL_dp_general * dp_dz
dL_dz_simple = p-y
print(f"via chain rule = ({dL_dp_general:.8f})({dp_dz:.8f}) = {dL_dz_chain:.8f}")
print(f"simplified p-y = {dL_dz_simple:.8f}")
assert np.allclose(dL_dz_chain, dL_dz_simple)


via chain rule = (-1.40656966)(0.20550031) = -0.28905050
simplified p-y = -0.28905050


## 5. Derive gradients for $w_1,w_2,b$

Recall

$$z=x_1w_1+x_2w_2+b.$$

Holding everything else constant:

$$\frac{\partial z}{\partial w_1}=x_1,\qquad
\frac{\partial z}{\partial w_2}=x_2,\qquad
\frac{\partial z}{\partial b}=1.$$

Apply the chain rule along each path:

$$\frac{\partial\ell}{\partial w_1}
=\frac{\partial\ell}{\partial z}\frac{\partial z}{\partial w_1}
=(p-y)x_1,$$

$$\frac{\partial\ell}{\partial w_2}=(p-y)x_2,$$

$$\frac{\partial\ell}{\partial b}=p-y.$$

Substituting $x_1=2$, $x_2=-1$, and $p-y=-0.2890505$:

$$dw_1=(-0.2890505)(2)=-0.5781010,$$
$$dw_2=(-0.2890505)(-1)=+0.2890505,$$
$$db=-0.2890505.$$

Notice that $dw_1$ and $dw_2$ have opposite signs because the two input values have opposite signs.


In [6]:
dz = p-y
dw1 = dz*x1
dw2 = dz*x2
db = dz

rows = [
    ("w₁", "(p-y)x₁", f"({dz:.6f})({x1:.1f})", dw1),
    ("w₂", "(p-y)x₂", f"({dz:.6f})({x2:.1f})", dw2),
    ("b",  "p-y",      f"{dz:.6f}", db),
]
html_rows = "".join(
    f"<tr><td>{name}</td><td>{formula}</td><td>{substitution}</td><td>{value:.8f}</td></tr>"
    for name, formula, substitution, value in rows
)
display(HTML("""
<table style='border-collapse:collapse;font-family:Arial;min-width:640px'>
<thead><tr style='background:#e0f2fe'><th>parameter</th><th>formula</th><th>substitution</th><th>gradient</th></tr></thead>
<tbody>""" + html_rows + "</tbody></table><style>td,th{padding:7px 12px;border:1px solid #94a3b8;text-align:right}td:first-child{text-align:center}</style>"))


parameter,formula,substitution,gradient
w₁,(p-y)x₁,(-0.289050)(2.0),-0.57810099
w₂,(p-y)x₂,(-0.289050)(-1.0),0.28905050
b,p-y,-0.289050,-0.28905050


## 6. Perform one Gradient Descent update

Using learning rate $\eta=0.1$:

$$w_1^{new}=w_1-\eta dw_1,$$
$$w_2^{new}=w_2-\eta dw_2,$$
$$b^{new}=b-\eta db.$$

Substitution gives

$$w_1^{new}=0.30-0.1(-0.5781010)=0.3578101,$$
$$w_2^{new}=-0.20-0.1(0.2890505)=-0.2289051,$$
$$b^{new}=0.10-0.1(-0.2890505)=0.1289051.$$

If the derivation and learning rate are sensible, recomputing the forward pass should give a smaller loss.


In [7]:
learning_rate = 0.1
new_w1 = w1-learning_rate*dw1
new_w2 = w2-learning_rate*dw2
new_bias = bias-learning_rate*db
new_z = x1*new_w1 + x2*new_w2 + new_bias
new_p = float(sigmoid(new_z))
new_loss = float(bce(np.array([y]), np.array([new_p])))

print(f"before: w=({w1:.6f}, {w2:.6f}), b={bias:.6f}, p={p:.6f}, loss={loss:.6f}")
print(f"after:  w=({new_w1:.6f}, {new_w2:.6f}), b={new_bias:.6f}, p={new_p:.6f}, loss={new_loss:.6f}")
print(f"loss change = {new_loss-loss:.6f}")
assert new_loss < loss


before: w=(0.300000, -0.200000), b=0.100000, p=0.710950, loss=0.341154
after:  w=(0.357810, -0.228905), b=0.128905, p=0.745249, loss=0.294037
loss change = -0.047117


## 7. Verify all three parameter gradients numerically

For any parameter $\theta$,

$$\frac{\partial\ell}{\partial\theta}\approx
\frac{\ell(\theta+\epsilon)-\ell(\theta-\epsilon)}{2\epsilon}.$$

This check does not replace the proof. It tests whether the derived formula and code agree at a selected point.


In [8]:
def scalar_loss(w1_value, w2_value, b_value):
    probability = sigmoid(x1*w1_value + x2*w2_value + b_value)
    return float(bce(np.array([y]), np.array([probability])))

numeric_dw1 = (scalar_loss(w1+epsilon,w2,bias)-scalar_loss(w1-epsilon,w2,bias))/(2*epsilon)
numeric_dw2 = (scalar_loss(w1,w2+epsilon,bias)-scalar_loss(w1,w2-epsilon,bias))/(2*epsilon)
numeric_db = (scalar_loss(w1,w2,bias+epsilon)-scalar_loss(w1,w2,bias-epsilon))/(2*epsilon)

print(f"dw1 analytical={dw1:.10f} numerical={numeric_dw1:.10f}")
print(f"dw2 analytical={dw2:.10f} numerical={numeric_dw2:.10f}")
print(f"db  analytical={db:.10f} numerical={numeric_db:.10f}")
assert np.allclose([dw1,dw2,db], [numeric_dw1,numeric_dw2,numeric_db], atol=1e-8)


dw1 analytical=-0.5781009947 numerical=-0.5781009948
dw2 analytical=0.2890504974 numerical=0.2890504974
db  analytical=-0.2890504974 numerical=-0.2890504974


# Part II — Two examples in one mini-batch

## 8. Batch values

Keep the same parameters but use two examples:

$$X=\begin{bmatrix}2&-1\\-1&2\end{bmatrix},\qquad
Y=\begin{bmatrix}1\\0\end{bmatrix},\qquad
W=\begin{bmatrix}0.30\\-0.20\end{bmatrix},\qquad b=0.10.$$

Each row of $X$ is one example. The batch mean loss is

$$L=\frac{\ell^{(1)}+\ell^{(2)}}{2}.$$

Because of this mean, each row's output gradient is

$$d z_i=\frac{p_i-y_i}{2}.$$


In [9]:
X = np.array([[2.0,-1.0],[-1.0,2.0]])
Y = np.array([[1.0],[0.0]])
W = np.array([[0.30],[-0.20]])
b = np.array([0.10])

Z = X@W+b
P = sigmoid(Z)
L = bce(Y,P)
dZ = (P-Y)/len(X)

print("X =\n",X)
print("Z =\n",Z)
print("P =\n",P)
print("Y =\n",Y)
print("dZ=(P-Y)/2 =\n",dZ)
print("mean BCE =",L)


X =
 [[ 2. -1.]
 [-1.  2.]]
Z =
 [[ 0.9]
 [-0.6]]
P =
 [[0.7109495 ]
 [0.35434369]]
Y =
 [[1.]
 [0.]]
dZ=(P-Y)/2 =
 [[-0.14452525]
 [ 0.17717185]]
mean BCE = 0.3893209126089868


## 9. From individual contributions to matrix multiplication

For two features,

$$\frac{\partial L}{\partial w_1}=x_{11}dz_1+x_{21}dz_2,$$
$$\frac{\partial L}{\partial w_2}=x_{12}dz_1+x_{22}dz_2.$$

These two dot products are exactly

$$dW=X^\top dZ.$$

Shape proof:

$$X^\top:(2,2),\qquad dZ:(2,1),\qquad dW:(2,1).$$

For bias, the same scalar $b$ was added to both logits, so both paths contribute:

$$db=dz_1+dz_2=\sum_i dz_i.$$


In [10]:
dW = X.T@dZ
db_batch = np.sum(dZ,axis=0)

example_1_contribution = X[0].reshape(2,1)*dZ[0,0]
example_2_contribution = X[1].reshape(2,1)*dZ[1,0]

print("example 1 contribution to dW =\n",example_1_contribution)
print("example 2 contribution to dW =\n",example_2_contribution)
print("sum of contributions =\n",example_1_contribution+example_2_contribution)
print("matrix X.T @ dZ =\n",dW)
print("db =",db_batch)
assert np.allclose(example_1_contribution+example_2_contribution,dW)


example 1 contribution to dW =
 [[-0.2890505 ]
 [ 0.14452525]]
example 2 contribution to dW =
 [[-0.17717185]
 [ 0.35434369]]
sum of contributions =
 [[-0.46622234]
 [ 0.49886894]]
matrix X.T @ dZ =
 [[-0.46622234]
 [ 0.49886894]]
db = [0.0326466]


## 10. Why the two examples can partially cancel

One example may push a weight upward while another pushes it downward. Batch Gradient Descent averages these competing local requests. Cancellation does not mean that neither example matters; it means their first-order effects oppose each other at the current parameters.

Larger batches provide a more stable average estimate. Smaller stochastic batches are noisier but cheaper and may help explore a complex loss landscape.


## Compact derivation map

$$\boxed{x_1,x_2,w_1,w_2,b}\longrightarrow
z=x_1w_1+x_2w_2+b$$

$$z\longrightarrow p=\sigma(z)\longrightarrow
\ell=-y\log p-(1-y)\log(1-p)$$

$$\frac{\partial\ell}{\partial p}=\frac{p-y}{p(1-p)},\qquad
\frac{dp}{dz}=p(1-p)$$

$$\boxed{dz=p-y}$$

$$\boxed{dw_1=x_1dz},\qquad
\boxed{dw_2=x_2dz},\qquad
\boxed{db=dz}$$

For a mean batch:

$$\boxed{dZ=\frac{P-Y}{N}},\qquad
\boxed{dW=X^\top dZ},\qquad
\boxed{db=\sum_i dZ_i}.$$


## Summary

- With two inputs, the logit is $z=x_1w_1+x_2w_2+b$.
- Sigmoid contributes $p(1-p)$; BCE contributes $(p-y)/(p(1-p))$.
- Chain-rule cancellation yields $dz=p-y$.
- Each weight gradient equals its input times the shared output error signal.
- Bias gradient equals the output error signal because $\partial z/\partial b=1$.
- A mean batch averages output gradients, then $X^\top dZ$ sums feature-scaled contributions.
- Centered finite differences confirm all three analytical parameter gradients.
- Subtracting these gradients with a small learning rate reduces the example loss.
